In [ ]:
# Imports
from datetime import datetime
import json
import numpy as np
import pandas as pd

from pyspark.sql import functions as F
from pyspark.sql import types as T

# Parameters
MIN_RUNS_FOR_NOTEARS = 30
LAGS = 1  # start with lag-1
CORR_THRESHOLD = 0.3
# Safety guard: max runs to pivot in Spark to avoid heavy full-history scans
MAX_RUNS_TO_PIVOT = 180

# ----------------------------
# Human priors (domain constraints)
# ----------------------------
# Notes:
# - Metric naming convention used across notebooks prefixes metrics with stage names: raw_*, bronze_*, silver_*
# - We encode two kinds of priors:
#   * Hard blacklist: edges that are not possible by design (e.g., Silver -> Raw)
#   * Hard whitelist: edges we strongly believe must exist (e.g., Raw -> Bronze for survival_rate impacts)
# Edit these lists before running discovery or provide them via external files.

# Defaults (empty — we will try to ingest external files at runtime)
HUMAN_PRIOR_BLACKLIST = [
    # Disallow any downstream -> upstream edges (leave empty to auto-generate conservative stage-level rules)
    # Example explicit pair (uncomment to hard-code):
    # ("silver_fuel_per_100km", "raw_raw_input_record_count"),
]

HUMAN_PRIOR_WHITELIST = [
    # Force obvious forward-stage relationships if you want higher prior weight
    # Example explicit pair (uncomment to hard-code):
    # ("raw_raw_input_record_count", "bronze_bronze_output_rows"),
]

# Ingest external prior files if present. This allows maintaining priors outside the notebook.
# - HUMAN_PRIOR_WHITELIST.py should expose HUMAN_PRIOR_WHITELIST = [(from, to), ...]
# - HUMAN_PRIOR_BLACKLIST.py (optional) can expose HUMAN_PRIOR_BLACKLIST similarly
try:
    from HUMAN_PRIOR_WHITELIST import HUMAN_PRIOR_WHITELIST as FILE_WHITELIST
    if isinstance(FILE_WHITELIST, (list, tuple)):
        # replace only if the external file provides a list/tuple
        HUMAN_PRIOR_WHITELIST = list(FILE_WHITELIST)
    print(f'Loaded {len(HUMAN_PRIOR_WHITELIST)} whitelist entries from HUMAN_PRIOR_WHITELIST.py')
except Exception:
    # no external whitelist found — fine, continue
    pass

try:
    from HUMAN_PRIOR_BLACKLIST import HUMAN_PRIOR_BLACKLIST as FILE_BLACKLIST
    if isinstance(FILE_BLACKLIST, (list, tuple)):
        HUMAN_PRIOR_BLACKLIST = list(FILE_BLACKLIST)
    print(f'Loaded {len(HUMAN_PRIOR_BLACKLIST)} blacklist entries from HUMAN_PRIOR_BLACKLIST.py')
except Exception:
    # external blacklist not present; orchestration will auto-generate conservative rules if desired
    pass

# Programmatic helper: generate blacklist rules based on metric name prefixes
# Example usage inside the orchestration after `metrics_pdf` exists:
#    auto_blacklist = generate_stage_blacklist(metrics_pdf.columns)
#    HUMAN_PRIOR_BLACKLIST.extend([p for p in auto_blacklist if p not in HUMAN_PRIOR_BLACKLIST])

def generate_stage_blacklist(metric_cols):
    """Generate conservative blacklist pairs from metric column names.

    Rules encoded:
    - forbid any edge from `silver_` metrics to `bronze_` or `raw_` metrics
    - forbid any edge from `bronze_` metrics to `raw_` metrics

    metric_cols: iterable of metric column names (strings)
    returns: list of (from_metric, to_metric) tuples
    """
    blacklist = []
    cols = list(metric_cols)
    for a in cols:
        for b in cols:
            if a == b:
                continue
            if a.startswith('silver_') and (b.startswith('bronze_') or b.startswith('raw_')):
                blacklist.append((a, b))
            if a.startswith('bronze_') and b.startswith('raw_'):
                blacklist.append((a, b))
    return blacklist

# Human-readable explanation for review
HUMAN_PRIOR_EXPLANATION = {
    'blacklist_logic': 'Forbid Silver->Bronze/Raw and Bronze->Raw edges by default. Add specific exceptions manually if needed.',
    'whitelist_tip': 'If a particular upstream->downstream metric relationship must exist, add it to HUMAN_PRIOR_WHITELIST before running or provide it via HUMAN_PRIOR_WHITELIST.py.',
}

# HUMAN INTERVENTION POINT: 
# - Edit HUMAN_PRIOR_BLACKLIST and HUMAN_PRIOR_WHITELIST above to encode hard constraints before learning (or place them in external files).
# - Optionally call generate_stage_blacklist(metrics_pdf.columns) after loading the pivot to auto-populate conservative rules.
# - After the skeleton is learned, review/prune edges at the review/prior step in the orchestration cell.
#
    for a in cols:
        for b in cols:
            if a == b:
                continue
            if a.startswith('silver_') and (b.startswith('bronze_') or b.startswith('raw_')):
                blacklist.append((a, b))
            if a.startswith('bronze_') and b.startswith('raw_'):
                blacklist.append((a, b))
    return blacklist

# Human-readable explanation for review
HUMAN_PRIOR_EXPLANATION = {
    'blacklist_logic': 'Forbid Silver->Bronze/Raw and Bronze->Raw edges by default. Add specific exceptions manually if needed.',
    'whitelist_tip': 'If a particular upstream->downstream metric relationship must exist, add it to HUMAN_PRIOR_WHITELIST before running.'
}

# HUMAN INTERVENTION POINT: 
# - Edit HUMAN_PRIOR_BLACKLIST and HUMAN_PRIOR_WHITELIST above to encode hard constraints before learning.
# - Optionally call generate_stage_blacklist(metrics_pdf.columns) after loading the pivot to auto-populate conservative rules.
# - After the skeleton is learned, review/prune edges at the review/prior step in the orchestration cell.


In [ ]:
# Preprocessing helper functions (Pandas)
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler


def preprocess_metrics_matrix(df: pd.DataFrame, zscore=True, impute_strategy='median'):
    """Impute small gaps, add missingness indicators, scale numeric columns.
    Returns (df_clean, metadata)
    """
    meta = {}
    # Record missingness
    missing_frac = df.isna().mean().to_dict()
    meta['missing_fraction'] = missing_frac

    # Add missingness indicator columns for metrics with >0 missing
    miss_cols = [c for c, v in missing_frac.items() if v > 0]
    for c in miss_cols:
        df[f"missing_{c}"] = df[c].isna().astype(float)

    # Impute
    imputer = SimpleImputer(strategy=impute_strategy)
    imputed = pd.DataFrame(imputer.fit_transform(df), index=df.index, columns=df.columns)

    # Scale
    if zscore:
        scaler = StandardScaler()
        scaled = pd.DataFrame(scaler.fit_transform(imputed), index=imputed.index, columns=imputed.columns)
    else:
        scaled = imputed

    return scaled, meta

In [ ]:
# Lag features

def add_lags(df: pd.DataFrame, lags=1, rolling_windows=None):
    out = df.copy()
    for lag in range(1, lags+1):
        shifted = df.shift(lag)
        shifted.columns = [f"{c}_lag{lag}" for c in df.columns]
        out = pd.concat([out, shifted], axis=1)
    if rolling_windows:
        for w in rolling_windows:
            rolled = df.rolling(window=w, min_periods=1).mean()
            rolled.columns = [f"{c}_rw{w}" for c in df.columns]
            out = pd.concat([out, rolled], axis=1)
    return out

In [ ]:
# partial-correlation helper removed — using causal-learn PC implementation

def partial_corr_matrix(df: pd.DataFrame):
    raise RuntimeError("partial_corr_matrix removed; use causal-learn PC via run_constraint_discovery")


In [ ]:
# quick_pc_skeleton fallback removed (causal-learn PC will be used)

def quick_pc_skeleton(*args, **kwargs):
    raise RuntimeError("quick_pc_skeleton removed; using causal-learn PC")


In [ ]:
# Run PC using causal-learn

def run_constraint_discovery(df: pd.DataFrame, alpha=0.05):
    """Run PC using causal-learn and return the PC object.

    Note: `pc` output is returned in `pc_object`. Converting to a list of
    edges depends on the downstream representation the user prefers; the
    returned `pc_object` can be inspected or stringified in artifacts.
    """
    print('Running PC from causal-learn (ensure variables are numeric and non-constant).')
    try:
        data = df.values
        pc_obj = pc(data, alpha=alpha)
        return {'method': 'causal-learn-pc', 'pc_object': pc_obj, 'edges': None}
    except Exception as e:
        return {'method': 'causal-learn-pc-error', 'error': str(e), 'edges': None}


In [ ]:
# NOTEARS (optional)

def run_notears(df: pd.DataFrame, l1=0.1, max_iter=100):
    if not has_notears:
        return {'method': 'notears-unavailable', 'adj': None}
    X = df.values.astype(float)
    # NOTE: user must install/adjust notears implementation used in their env
    W_est = notears(X, lambda1=l1, max_iter=max_iter)
    return {'method': 'notears', 'adj': W_est}

In [ ]:
# Granger causality checks for candidate edges

def granger_check(time_df: pd.DataFrame, candidate_pairs, maxlag=1):
    results = {}
    if not has_statsmodels:
        print('statsmodels not available; skipping Granger checks')
        return results
    for a, b, score in candidate_pairs:
        # Build two-column series [b, a] as statsmodels expects endog
        try:
            test_df = time_df[[b, a]].dropna()
            if len(test_df) < maxlag + 5:
                continue
            res = grangercausalitytests(test_df, maxlag=maxlag, verbose=False)
            pvals = [res[i+1][0]['ssr_ftest'][1] for i in range(maxlag)]
            results[(a,b)] = {'pvalues': pvals, 'min_p': float(np.min(pvals))}
        except Exception as e:
            results[(a,b)] = {'error': str(e)}
    return results

In [ ]:
# Human priors application and visualization helpers

import networkx as nx
import matplotlib.pyplot as plt


def apply_human_priors(skeleton_edges, blacklist=HUMAN_PRIOR_BLACKLIST, whitelist=HUMAN_PRIOR_WHITELIST):
    """Apply simple human priors to the skeleton edges.

    - `skeleton_edges` expected as list of tuples (a,b,score)
    - `blacklist` is list of (from, to) tuples to remove
    - `whitelist` is list of (from, to) tuples to force present

    Returns filtered_edges, and a dict with actions taken for review.
    """
    kept = []
    removed = []
    present = set((a,b) for a,b,_ in skeleton_edges)

    # Remove blacklisted edges
    for a,b,score in skeleton_edges:
        if (a,b) in blacklist or (a,b) in [(x,y) for x,y in blacklist]:
            removed.append((a,b,score,'blacklisted'))
        else:
            kept.append((a,b,score))

    # Add whitelisted edges if missing (score=None)
    added = []
    for a,b in whitelist:
        if (a,b) not in present:
            added.append((a,b,None,'whitelisted'))
            kept.append((a,b,None))

    review = {'removed': removed, 'added': added}
    return kept, review


def visualize_skeleton(edges, path="/dbfs/tmp/causal_skeleton.png", top_k=100):
    """Create a simple networkx plot of skeleton edges and save to path.
    Edges is list of (a,b,score). We limit nodes/edges shown to top_k by abs(score).
    """
    try:
        if not edges:
            print('No edges to visualize')
            return None
        # Sort by absolute score when available
        scored = [e for e in edges if e[2] is not None]
        scored.sort(key=lambda x: -abs(x[2]))
        show_edges = scored[:top_k] if scored else edges[:top_k]

        G = nx.DiGraph()
        for a,b,score in show_edges:
            w = 1.0 if score is None else float(score)
            G.add_edge(a, b, weight=w)

        plt.figure(figsize=(14, 10))
        pos = nx.spring_layout(G, k=0.5, iterations=50)
        weights = [abs(G[u][v]['weight']) for u,v in G.edges()]
        nx.draw(G, pos, with_labels=True, node_size=400, font_size=8, width=[max(0.5, w*2) for w in weights])
        plt.title('Causal skeleton (top edges)')
        plt.tight_layout()
        # Save to DBFS if path starts with /dbfs, else local
        plt.savefig(path, dpi=150)
        plt.close()
        print(f'Wrote skeleton visualization to {path}')
        return path
    except Exception as e:
        print('Visualization failed:', e)
        return None

# HUMAN INTERVENTION POINTS (explicit):
# 1) Before running structure learning: edit HUMAN_PRIOR_BLACKLIST / WHITELIST above.
# 2) After skeleton is produced: call apply_human_priors(skeleton['edges'], ...) and review returned `review` dict.
# 3) After pruning/whitelisting: visualize with visualize_skeleton(filtered_edges) and manually prune further.
# 4) For intervention tests: create synthetic runs (see design doc) and re-run the pipeline to validate RCA.


In [ ]:
# Example orchestration function to run steps 1-5

def run_baseline_pipeline(spark, run_all=True, max_runs=MAX_RUNS_TO_PIVOT):
    print('Loading metrics (step 1)')
    metrics_pdf = spark_metrics_to_matrix(spark, metrics_table=METRICS_TABLE, max_runs=max_runs)

    # Auto-populate conservative blacklist from metric prefixes
    try:
        auto_blacklist = generate_stage_blacklist(metrics_pdf.columns)
        HUMAN_PRIOR_BLACKLIST.extend([p for p in auto_blacklist if p not in HUMAN_PRIOR_BLACKLIST])
        print(f'Auto-added {len(auto_blacklist)} blacklist pairs based on stage prefixes')
    except Exception as e:
        print('Auto blacklist generation failed:', e)

    # Optionally import an external whitelist file if present and extend runtime whitelist
    try:
        from HUMAN_PRIOR_WHITELIST import HUMAN_PRIOR_WHITELIST as FILE_WHITELIST
        HUMAN_PRIOR_WHITELIST.extend([p for p in FILE_WHITELIST if p not in HUMAN_PRIOR_WHITELIST])
        print(f'Extended whitelist from HUMAN_PRIOR_WHITELIST.py with {len(FILE_WHITELIST)} entries')
    except Exception:
        # not required; continue
        pass

    print('Preprocessing (step 2)')
    scaled, meta = preprocess_metrics_matrix(metrics_pdf)

    print('Adding lags (step 3)')
    lagged = add_lags(scaled, lags=LAGS, rolling_windows=[3])

    print('Filtering columns with low variance or constant values')
    nunique = lagged.nunique()
    keep_cols = nunique[nunique > 1].index.tolist()
    lagged = lagged[keep_cols]

    print('Computing correlation / skeleton (step 4)')
    skeleton_res = run_constraint_discovery(lagged)

    # Extract edges in a robust way: prefer causal-learn PC output, fallback to correlation threshold
    skeleton_edges = []
    if skeleton_res.get('pc_object') is not None:
        pc_obj = skeleton_res['pc_object']
        try:
            # Try common causal-learn representations
            # 1) networkx graph in pc_obj.G (nodes may be indexed)
            if hasattr(pc_obj, 'G'):
                G = getattr(pc_obj, 'G')
                try:
                    # networkx graph
                    for u, v in G.edges():
                        # If nodes are integers, map to column names; otherwise assume names
                        a = lagged.columns[u] if isinstance(u, int) and u < len(lagged.columns) else str(u)
                        b = lagged.columns[v] if isinstance(v, int) and v < len(lagged.columns) else str(v)
                        skeleton_edges.append((a, b, None))
                except Exception:
                    pass
            # 2) some pc objects expose a `skeleton` list of index pairs
            if not skeleton_edges and hasattr(pc_obj, 'skeleton'):
                sk = getattr(pc_obj, 'skeleton')
                try:
                    for i, j in sk:
                        a = lagged.columns[int(i)]
                        b = lagged.columns[int(j)]
                        skeleton_edges.append((a, b, None))
                except Exception:
                    pass
        except Exception:
            skeleton_edges = []

    # Fallback: use absolute Pearson correlation threshold to create undirected candidate edges
    if not skeleton_edges:
        try:
            corr = lagged.corr().abs()
            cols = corr.columns.tolist()
            for i, a in enumerate(cols):
                for j in range(i+1, len(cols)):
                    b = cols[j]
                    val = float(corr.loc[a, b])
                    if val >= CORR_THRESHOLD:
                        skeleton_edges.append((a, b, val))
            skeleton_res['method'] = skeleton_res.get('method', 'fallback-corr')
            skeleton_res['edges'] = skeleton_edges
            print(f'Fallback correlation generated {len(skeleton_edges)} candidate edges (threshold={CORR_THRESHOLD})')
        except Exception as e:
            print('Failed to build fallback correlation skeleton:', e)

    # HUMAN REVIEW POINT: apply human priors to skeleton before further processing
    filtered_edges, review = apply_human_priors(skeleton_edges, blacklist=HUMAN_PRIOR_BLACKLIST, whitelist=HUMAN_PRIOR_WHITELIST)

    print('\n=== HUMAN REVIEW ===')
    print('Edges removed by blacklist:', len(review.get('removed', [])))
    print('Edges added by whitelist:', len(review.get('added', [])))
    print('Please review `review` and `filtered_edges` before proceeding to NOTEARS or Granger tests.')

    # Visualize filtered skeleton for quick review
    viz_path = visualize_skeleton(filtered_edges, path='/dbfs/tmp/causal_skeleton.png')

    # Candidate pairs from skeleton - use filtered edges
    candidate_pairs = filtered_edges

    print('Running Granger checks on candidate edges')
    granger_res = granger_check(metrics_pdf, candidate_pairs, maxlag=LAGS)

    print('Optional NOTEARS (step 5)')
    notears_res = None
    # Only run NOTEARS if we have enough rows (after lagging) and library available
    if len(lagged) >= MIN_RUNS_FOR_NOTEARS and has_notears:
        try:
            notears_res = run_notears(lagged)
        except Exception as e:
            notears_res = {'error': str(e)}
    else:
        notears_res = {'skipped': f'Not enough runs for NOTEARS or library missing (have {len(lagged)}, need {MIN_RUNS_FOR_NOTEARS})'}

    # Save artifacts
    try:
        lagged.to_parquet(OUT_PARQUET)
    except Exception:
        # fallback to local
        lagged.to_parquet('./tmp/causal_metrics_matrix.parquet')

    artifacts = {'skeleton': skeleton_res, 'filtered_edges': filtered_edges, 'review': review, 'notears': notears_res, 'granger': granger_res, 'meta': meta, 'viz_path': viz_path}
    try:
        with open('/dbfs/tmp/causal_artifacts.json','w') as f:
            json.dump(artifacts, f, default=str)
    except Exception:
        with open('./tmp/causal_artifacts.json','w') as f:
            json.dump(artifacts, f, default=str)

    print('\nBaseline run complete. Artifacts saved to /dbfs/tmp/ (or ./tmp/).')
    return artifacts

# On Databricks, run:
# artifacts = run_baseline_pipeline(spark)


**Design Document (concise)**

- Purpose: build a fast baseline causal discovery pipeline to detect DQ anomalies and provide candidate root causes.
- Inputs: daily `temp_raw_metrics` table (date, pipeline_stage, metric_name, metric_value)
- Output: skeleton graph, optional weighted DAG, Granger-supported edges, artifacts saved to `/dbfs/tmp/`.

Key parameters (defaults):
- `LAGS = 1` (start with 1-day lag)
- `CORR_THRESHOLD = 0.3` (edge candidate threshold)
- `MIN_RUNS_FOR_NOTEARS = 30` (practical lower bound)
- `BOOTSTRAP = 100` (optional later)

Algorithmic steps:
1. Pivot metrics to wide matrix; impute and z-score.
2. Add lag-1 and rolling features.
3. Build skeleton via partial-correlation or PC (if available).
4. Run NOTEARS for weighted DAG (optional if enough data).
5. Validate/ orient edges with Granger tests.
6. Save artifacts and visualize graph externally (e.g., `networkx`, `pyvis`).

Next actions after baseline:
- Bootstrap stability scores; encode human priors; create RCA scoring and injection tests.

References: ExplainIt! (2019), Yang et al. (2022), DataExposer (2021).